# 04b — Events Assembly: Extractie, Verwerking & MAD-Integratie
## Masterproef: Spatiotemporal Prediction and Optimization of Car Parking in Mechelen

---

### Academische motivatie

De integratie van evenementdata in parkeermodellen is empirisch goed
onderbouwd. Modellen die weerdata combineren met evenement-indicatoren
behalen significant hogere nauwkeurigheid dan weerpuur modellen:
de PewLSTM (Periodic Weather-Aware LSTM with Event Mechanism) haalt
93.84% nauwkeurigheid door het expliciet modelleren van evenement-
patronen bovenop weerdata (Zhang et al., 2021). Fokker et al. (2021)
tonen aan dat evenement-variabelen kritische predictors zijn in
langetermijn parkeerbezettingsmodellen met SARIMAX-structuur.
Chang et al. (2022) bevestigen dat externe factoren — waaronder
evenementen — de predictieve kracht van gedeeld-transport modellen
substantieel verbeteren.

**Bijzonder relevant voor Mechelen**: de stad combineert een compact
centrum (alle parkings binnen 800m van elkaar) met een divers
evenementenprofiel: grootschalige festivals (Maanrock: ~100.000
bezoekers), tradities (Hanswijkprocessie: ~50.000 bezoekers),
en reguliere voetbalwedstrijden (AFAS-stadion: ~15.000 capaciteit).
In een compacte stad heeft elke schaalklasse van evenement een
potentieel merkbare en meetbare impact op de totale parkeerdruk.

### Afbakening

Dit notebook behandelt uitsluitend evenementen met **mechanistische
link naar stedelijke parkeerdruk** in het centrum van Mechelen:

| Categorie | Geselecteerd | Motivering |
|---|---|---|
| Maanrock | ✓ | ~100.000 bezoekers, 4 daags, centrum |
| Hanswijkprocessie | ✓ | ~50.000 bezoekers, stadscentrum |
| KV Mechelen thuiswedstrijden | ✓ | ~8.000–15.000 toeschouwers, nabij centrum |
| Carnavalsfoor & -stoet | ✓ | Centrum-Botermarkt, verhoogde parkeerdruk |
| Zomerkermis (Grote Markt) | ✓ | Stadscentrum, 2,5 week |
| Herfstkermis (Veemarkt) | ✓ | Stadscentrum, 2,5 week |
| Wijkkermissen (Leest, Heffen...) | ✗ | Geen centrum-impact (Stad Mechelen, 2025) |
| Concerten Nekkerhal | ✗ | Geen historische data beschikbaar |

### Output

| Bestand | Locatie | Granulariteit |
|---|---|---|
| `events_master.parquet` | `data_intermediate/` | Dagelijks |
| `MAD_shortterm.parquet` (update) | `data_processed/` | Uurlijks |
| `MAD_longterm.parquet` (update) | `data_processed/` | Uurlijks |


## Databronnen & kwaliteitsclassificatie

Eventdata heeft een fundamenteel andere kwaliteits-keten dan
observationele sensordata: ze zijn historisch schaars en
verspreid over primaire bronnen (officiële websites, Wikipedia,
stadsdocumenten) en secondaire bronnen (sportstatistieken-sites).
We classificeren per bron:

| Bron | Bron-type | Betrouwbaarheid | Volledigheid | Strategie |
|---|---|---|---|---|
| Stad Mechelen (kermissen 2025) | Primair | Hoog | Enkel 2025 | Hardcode + geschatte historische patroon |
| Hanswijkbasiliek.com | Primair | Hoog | 2022–2026 | Hardcode; 2020–2021 COVID-flag |
| Wikipedia KVM-seizoenspagina's | Secundair | Hoog | 2020–2025 | Geautomatiseerde extractie per seizoen |
| Maanrock.be / Wikipedia | Secundair | Hoog | 2020–2026 | Hardcode (COVID-uitzonderingen) |
| mechelen.be/data-kermissen | Primair archief | Matig | Variabel | Geschat patroon ± verifieer |

### Kwantificatie van onzekerheid
Data die **geschat** zijn (niet rechtstreeks van primaire bron)
worden gelabeld met een `data_confidence`-kolom:
- `verified` — rechtstreeks van primaire bron
- `estimated` — afgeleid uit stabiel patroon, niet bevestigd
- `covid_modified` — evenement vond geen/afgeslankte doorgang
  wegens COVID-19 maatregelen (2020, 2021)


In [1]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import re
import time
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from mechelen_parking import get_default_paths

PATHS = get_default_paths()
ROOT = PATHS.root
DATA_INT = ROOT / "data_intermediate"
DATA_PROC = PATHS.data_processed

PROJECT_START = pd.Timestamp("2019-01-01")
PROJECT_END = pd.Timestamp("2026-02-03")

print(f"Project datumbereik: {PROJECT_START.date()} → {PROJECT_END.date()}")
print(f"DATA_INT  : {DATA_INT}")
print(f"DATA_PROC : {DATA_PROC}")


Project datumbereik: 2019-01-01 → 2026-02-03
DATA_INT  : /Users/emilevandevoorde/Documents/mechelen_parking/data_intermediate
DATA_PROC : /Users/emilevandevoorde/Documents/mechelen_parking/data_processed


## Stap 1 — Basisstructuur & hulpfuncties

Alle evenementen worden gemodelleerd op **dagelijks niveau**
en bevatten de volgende velden:

| Kolom | Dtype | Beschrijving |
|---|---|---|
| `date` | datetime64 | Datum van het evenement (één rij per dag) |
| `event_name` | str | Leesbare naam |
| `event_type` | str | Categorie (football / festival / procession / carnival / kermis) |
| `event_scale` | int8 | 1=klein, 2=matig, 3=groot (gebaseerd op verwachte bezoekers) |
| `expected_attendance` | float | Geschatte bezoekers (NaN indien onbekend) |
| `is_multiday_event` | int8 | 1 indien onderdeel van meerداgsevenement |
| `football_kickoff_hour` | float | Uur van aftrap (alleen voor football, anders NaN) |
| `data_confidence` | str | verified / estimated / covid_modified |

### Evenement-schaal definitie

Gebaseerd op verwachte parkeerimpact (niet pure bezoekersaantallen):

| Schaal | Label | Bereik bezoekers | Voorbeelden |
|---|---|---|---|
| 1 | Klein | < 5.000 | Wijkkermis, carnavalsstoet |
| 2 | Matig | 5.000–30.000 | KVM thuiswedstrijd, Hanswijkprocessie, kermis centrum |
| 3 | Groot | > 30.000 | Maanrock |


In [3]:
def make_event_records(
    dates,
    event_name: str,
    event_type: str,
    event_scale: int,
    expected_attendance: float = np.nan,
    football_kickoff_hour: float = np.nan,
    data_confidence: str = "verified",
    is_multiday_event: int = 0,
) -> pd.DataFrame:
    """
    Maakt een gestandaardiseerd event-DataFrame aan voor een lijst van datums.
    Accepteert strings ("YYYY-MM-DD"), Timestamps of datum-ranges.
    """
    records = []
    for d in dates:
        records.append({
            "date"                  : pd.Timestamp(d).normalize(),
            "event_name"            : event_name,
            "event_type"            : event_type,
            "event_scale"           : int(event_scale),
            "expected_attendance"   : expected_attendance,
            "football_kickoff_hour" : football_kickoff_hour,
            "data_confidence"       : data_confidence,
            "is_multiday_event"     : int(is_multiday_event),
        })
    return pd.DataFrame(records)


def date_range_list(start: str, end: str) -> list:
    """Geeft lijst van alle datums (inclusief start en end)."""
    return pd.date_range(start=start, end=end, freq="D").tolist()


print("✓ Hulpfuncties geladen")


✓ Hulpfuncties geladen


## Stap 2 — Maanrock

Maanrock is het grootste jaarlijkse stadsfeest van Mechelen en vindt
traditioneel plaats op het **laatste weekend van augustus** over 4 dagen
(donderdag t/m zondag). Met ~100.000 bezoekers over het volledige
weekend (editie 2023) is dit verreweg het meest impactvolle evenement
voor stedelijke parkeerdruk.

### Databron & COVID-behandeling

| Jaar | Status | Datums | Bron | Confidence |
|---|---|---|---|---|
| 2020 | ❌ Geannuleerd | — | COVID-19 | covid_modified |
| 2021 | ⚠️ Beperkte editie "in het park" | ~28 aug | Beperkt publiek | covid_modified |
| 2022 | ✓ Volledig | 25–28 aug | Maanrock.be / Wikipedia | verified |
| 2023 | ✓ Volledig | 24–27 aug | Maanrock.be | verified |
| 2024 | ✓ Volledig | 22–25 aug | Maanrock.be | verified |
| 2025 | ✓ Volledig | 21–24 aug | Maanrock.be | verified |
| 2026 | ✓ Gepland | 27–30 aug | Maanrock.be | verified |

**Modelleeraanbeveling voor COVID-jaren**: de editie 2021 had sterk
beperkte capaciteit en geen vergelijkbaar parkeereffect met een
normale editie. De `covid_modified` flag laat downstream modellen
toe deze rijen apart te behandelen of te maskeren.


In [4]:
maanrock_events = []

# 2020: geannuleerd → geen evenement toegevoegd
# (parkeerimpact = 0 → geen rij nodig; NaN = geen event, niet anders)

# 2021: beperkte editie — wel rij maar covid_modified flag
maanrock_events.append(make_event_records(
    dates             = date_range_list("2021-08-28", "2021-08-29"),
    event_name        = "Maanrock (beperkte editie)",
    event_type        = "festival",
    event_scale       = 1,          # sterk afgeschaald
    expected_attendance = 2000,
    data_confidence   = "covid_modified",
    is_multiday_event = 1,
))

# 2022: 25–28 augustus
maanrock_events.append(make_event_records(
    dates             = date_range_list("2022-08-25", "2022-08-28"),
    event_name        = "Maanrock",
    event_type        = "festival",
    event_scale       = 3,
    expected_attendance = 80000,
    data_confidence   = "verified",
    is_multiday_event = 1,
))

# 2023: 24–27 augustus (100.000 bezoekers)
maanrock_events.append(make_event_records(
    dates             = date_range_list("2023-08-24", "2023-08-27"),
    event_name        = "Maanrock",
    event_type        = "festival",
    event_scale       = 3,
    expected_attendance = 100000,
    data_confidence   = "verified",
    is_multiday_event = 1,
))

# 2024: 22–25 augustus
maanrock_events.append(make_event_records(
    dates             = date_range_list("2024-08-22", "2024-08-25"),
    event_name        = "Maanrock",
    event_type        = "festival",
    event_scale       = 3,
    expected_attendance = 90000,
    data_confidence   = "verified",
    is_multiday_event = 1,
))

# 2025: 21–24 augustus
maanrock_events.append(make_event_records(
    dates             = date_range_list("2025-08-21", "2025-08-24"),
    event_name        = "Maanrock",
    event_type        = "festival",
    event_scale       = 3,
    expected_attendance = 90000,
    data_confidence   = "verified",
    is_multiday_event = 1,
))

# 2026: 27–30 augustus
maanrock_events.append(make_event_records(
    dates             = date_range_list("2026-08-27", "2026-08-30"),
    event_name        = "Maanrock",
    event_type        = "festival",
    event_scale       = 3,
    expected_attendance = 90000,
    data_confidence   = "verified",
    is_multiday_event = 1,
))

df_maanrock = pd.concat(maanrock_events, ignore_index=True)
print(f"Maanrock: {len(df_maanrock)} dag-records over {df_maanrock['date'].dt.year.nunique()} jaren")
print(df_maanrock[["date","event_name","event_scale","data_confidence"]].to_string(index=False))


Maanrock: 22 dag-records over 6 jaren
      date                 event_name  event_scale data_confidence
2021-08-28 Maanrock (beperkte editie)            1  covid_modified
2021-08-29 Maanrock (beperkte editie)            1  covid_modified
2022-08-25                   Maanrock            3        verified
2022-08-26                   Maanrock            3        verified
2022-08-27                   Maanrock            3        verified
2022-08-28                   Maanrock            3        verified
2023-08-24                   Maanrock            3        verified
2023-08-25                   Maanrock            3        verified
2023-08-26                   Maanrock            3        verified
2023-08-27                   Maanrock            3        verified
2024-08-22                   Maanrock            3        verified
2024-08-23                   Maanrock            3        verified
2024-08-24                   Maanrock            3        verified
2024-08-25              

## Stap 3 — Hanswijkprocessie

De Hanswijkprocessie is een UNESCO-erkend cultureel erfgoed en trekt
jaarlijks tienduizenden bezoekers naar het stadscentrum op de zondag
voor Hemelvaartsdag. Het is een ééndag evenement maar met impact
vergelijkbaar met een grote voetbalwedstrijd qua parkeerdruk.

De processe trekt door het centrum van Mechelen, wat directe impact
heeft op parkeertoegankelijkheid én op de vraag naar randparkings.

### COVID-behandeling

| Jaar | Status | Datum | Confidence |
|---|---|---|---|
| 2020 | ❌ Geannuleerd | — | covid_modified |
| 2021 | ⚠️ Beperkte editie | ~23 mei | covid_modified |
| 2022 | ✓ | 29 mei | verified |
| 2023 | ✓ | 21 mei | verified |
| 2024 | ✓ | 12 mei | verified |
| 2025 | ✓ | 25 mei | verified |
| 2026 | ✓ | 10 mei | verified |


In [5]:
hanswijkprocessie_data = [
    # (datum,        confidence,       attendance)
    ("2021-05-23",   "covid_modified", 5000),
    ("2022-05-29",   "verified",       50000),
    ("2023-05-21",   "verified",       50000),
    ("2024-05-12",   "verified",       50000),
    ("2025-05-25",   "verified",       50000),
    ("2026-05-10",   "verified",       50000),
]

hanswijkprocessie_records = []
for datum, conf, att in hanswijkprocessie_data:
    hanswijkprocessie_records.append(make_event_records(
        dates               = [datum],
        event_name          = "Hanswijkprocessie",
        event_type          = "procession",
        event_scale         = 2,
        expected_attendance = att,
        data_confidence     = conf,
        is_multiday_event   = 0,
    ))

df_hanswijkprocessie = pd.concat(hanswijkprocessie_records, ignore_index=True)
print(f"Hanswijkprocessie: {len(df_hanswijkprocessie)} records")
print(df_hanswijkprocessie[["date","event_scale","data_confidence"]].to_string(index=False))


Hanswijkprocessie: 6 records
      date  event_scale data_confidence
2021-05-23            2  covid_modified
2022-05-29            2        verified
2023-05-21            2        verified
2024-05-12            2        verified
2025-05-25            2        verified
2026-05-10            2        verified


## Stap 4 — Carnavalsfoor & Carnavalsstoet

Het carnaval van Mechelen omvat twee componenten met elk eigen
parkeerimpact:

1. **Carnavalsstoet** (1 dag, zondag): verhoogde parkeerdruk door
   bezoekers van buiten de stad; vergelijkbaar met een kleine-medium
   voetbalwedstrijd qua parkeervolume
2. **Carnavalfoor** (5-6 dagen, Botermarkt): dagelijkse verhoogde druk
   in het stadscentrum gedurende de gehele foorperiode; kenmerkend
   door verhoogde avondparkering

### Patroon
Het carnaval valt **altijd op het laatste weekend van maart**
(stoet = zondag, foor = zaterdag daarvoor t/m woensdag daarna).
2020 en 2021 waren beperkt/geannuleerd wegens COVID.

| Jaar | Foor (van–tot) | Stoet | Confidence |
|---|---|---|---|
| 2020 | — | — | covid_modified |
| 2021 | — | — | covid_modified |
| 2022 | ~26–30 mrt | ~27 mrt | estimated |
| 2023 | ~25–29 mrt | ~26 mrt | estimated |
| 2024 | ~30 mrt – 3 apr | ~31 mrt | estimated |
| 2025 | 29 mrt – 2 apr | 30 mrt | verified |
| 2026 | 28 mrt – 1 apr | 29 mrt | verified |

> ⚠️ **Verificatieplicht**: historische foor-data (2022–2024) zijn
> **geschat** op basis van het stabiele patroon. Verificeer via
> mechelen.be/data-kermissen archief of Wayback Machine voor
> exacte datums.


In [6]:
carnaval_config = [
    # (foor_start,   foor_end,     stoet_datum,   confidence)
    ("2022-03-26",  "2022-03-30",  "2022-03-27",  "estimated"),
    ("2023-03-25",  "2023-03-29",  "2023-03-26",  "estimated"),
    ("2024-03-30",  "2024-04-03",  "2024-03-31",  "estimated"),
    ("2025-03-29",  "2025-04-02",  "2025-03-30",  "verified"),
    ("2026-03-28",  "2026-04-01",  "2026-03-29",  "verified"),
]

carnaval_records = []

for foor_start, foor_end, stoet_datum, conf in carnaval_config:
    year = pd.Timestamp(foor_start).year

    # Foor: alle dagen (scale 1 — dagelijkse achtergronddruk)
    carnaval_records.append(make_event_records(
        dates               = date_range_list(foor_start, foor_end),
        event_name          = f"Carnavalfoor {year}",
        event_type          = "carnival",
        event_scale         = 1,
        expected_attendance = np.nan,
        data_confidence     = conf,
        is_multiday_event   = 1,
    ))

    # Stoet: één dag, hogere schaal (extra bezoekers)
    carnaval_records.append(make_event_records(
        dates               = [stoet_datum],
        event_name          = f"Carnavalsstoet {year}",
        event_type          = "carnival",
        event_scale         = 2,
        expected_attendance = 20000,
        data_confidence     = conf,
        is_multiday_event   = 0,
    ))

df_carnaval = pd.concat(carnaval_records, ignore_index=True)

# Dedupliceer stoet-datum: die staat al in foor EN als aparte stoet-rij
# → bewaar beide want event_type en event_scale zijn anders
#   (in consolidatie worden per datum ALLE events samengevoegd)
print(f"Carnaval: {len(df_carnaval)} dag-records")
print(df_carnaval[["date","event_name","event_scale","data_confidence"]]
      .to_string(index=False))


Carnaval: 30 dag-records
      date          event_name  event_scale data_confidence
2022-03-26   Carnavalfoor 2022            1       estimated
2022-03-27   Carnavalfoor 2022            1       estimated
2022-03-28   Carnavalfoor 2022            1       estimated
2022-03-29   Carnavalfoor 2022            1       estimated
2022-03-30   Carnavalfoor 2022            1       estimated
2022-03-27 Carnavalsstoet 2022            2       estimated
2023-03-25   Carnavalfoor 2023            1       estimated
2023-03-26   Carnavalfoor 2023            1       estimated
2023-03-27   Carnavalfoor 2023            1       estimated
2023-03-28   Carnavalfoor 2023            1       estimated
2023-03-29   Carnavalfoor 2023            1       estimated
2023-03-26 Carnavalsstoet 2023            2       estimated
2024-03-30   Carnavalfoor 2024            1       estimated
2024-03-31   Carnavalfoor 2024            1       estimated
2024-04-01   Carnavalfoor 2024            1       estimated
2024-04-02   Ca

## Stap 5 — Kermissen (centrum-relevant)

Stad Mechelen organiseert jaarlijks 12 kermissen, waarvan uitsluitend
de twee stadscentrum-kermissen relevant zijn voor de parkeerdataset:

- **Zomerkermis – Grote Markt** (± 2,5 week, late juni – mid juli)
- **Herfstkermis – Veemarkt** (± 2,5 week, vroeg oktober – laat oktober)

Wijkkermissen (Leest, Heffen, Muizen, Hombeek, Walem) zijn
uitgesloten op basis van de locatie buiten het parkeerinzamelgebied
en de verwaarloosbare centrumimpact.

### Patroon-gebaseerde schatting historische data

De kermisdata van 2025 zijn officieel gepubliceerd (mechelen.be).
Voor 2020–2024 geldt een **stabiel jaarlijks patroon** dat
verschuift met ~1 week per jaar afhankelijk van het weekdagpatroon.
Deze schattingen zijn als `estimated` geflagd en dienen geverifieerd
te worden via het kermisarchief op mechelen.be of de Wayback Machine.

| Jaar | Zomerkermis Grote Markt | Herfstkermis Veemarkt | Confidence |
|---|---|---|---|
| 2020 | 24 jun – 10 jul | 2 okt – 18 okt | estimated |
| 2021 | 23 jun – 9 jul | 1 okt – 17 okt | estimated |
| 2022 | 29 jun – 15 jul | 30 sep – 16 okt | estimated |
| 2023 | 28 jun – 14 jul | 6 okt – 22 okt | estimated |
| 2024 | 26 jun – 12 jul | 4 okt – 20 okt | estimated |
| 2025 | 25 jun – 11 jul | 3 okt – 19 okt | **verified** |


In [9]:
kermis_config = [
    # (jaar, zomer_start, zomer_end, herfst_start, herfst_end, confidence)
    (2020, "2020-06-24", "2020-07-10", "2020-10-02", "2020-10-18", "estimated"),
    (2021, "2021-06-23", "2021-07-09", "2021-10-01", "2021-10-17", "estimated"),
    (2022, "2022-06-29", "2022-07-15", "2022-09-30", "2022-10-16", "estimated"),
    (2023, "2023-06-28", "2023-07-14", "2023-10-06", "2023-10-22", "estimated"),
    (2024, "2024-06-26", "2024-07-12", "2024-10-04", "2024-10-20", "estimated"),
    (2025, "2025-06-25", "2025-07-11", "2025-10-03", "2025-10-19", "verified"),
]

kermis_records = []

for jaar, zs, ze, hs, he, conf in kermis_config:
    kermis_records.append(make_event_records(
        dates               = date_range_list(zs, ze),
        event_name          = f"Zomerkermis Grote Markt {jaar}",
        event_type          = "kermis",
        event_scale         = 2,
        expected_attendance = np.nan,
        data_confidence     = conf,
        is_multiday_event   = 1,
    ))
    kermis_records.append(make_event_records(
        dates               = date_range_list(hs, he),
        event_name          = f"Herfstkermis Veemarkt {jaar}",
        event_type          = "kermis",
        event_scale         = 2,
        expected_attendance = np.nan,
        data_confidence     = conf,
        is_multiday_event   = 1,
    ))

df_kermis = pd.concat(kermis_records, ignore_index=True)
print(f"Kermissen: {len(df_kermis)} dag-records over {df_kermis['date'].dt.year.nunique()} jaren")
print(f"  Zomerkermis gemiddelde duur : "
      f"{len(df_kermis[df_kermis['event_name'].str.contains('Zomer')]) / 6:.0f} dagen/jaar")
print(f"  Herfstkermis gemiddelde duur: "
      f"{len(df_kermis[df_kermis['event_name'].str.contains('Herfst')]) / 6:.0f} dagen/jaar")


Kermissen: 204 dag-records over 6 jaren
  Zomerkermis gemiddelde duur : 17 dagen/jaar
  Herfstkermis gemiddelde duur: 17 dagen/jaar


## Stap 6 — KV Mechelen thuiswedstrijden

KV Mechelen speelt thuiswedstrijden in het **AFAS-stadion Achter de
Kazerne** (capaciteit: ~15.000 plaatsen), op ~1 km van het stadscentrum.
Op basis van de parkeerlokalisatie (P Tinel, P Grote Markt, P Veemarkt)
en de loopafstand zijn stadionbezoekers waarschijnlijke gebruikers van
de centrumparking.

Het parkeereffect van voetbalwedstrijden is sterk afhankelijk van:
1. **Aanvangstijdstip** (middagwedstrijden vs. avondwedstrijden)
2. **Tegenstander** (topper vs. mindere tegenstander)
3. **Competitie** (league vs. bekerwedstrijd)

KV Mechelen speelde in de volledige projectperiode (2019–2026) in
de Jupiler Pro League (1A), de hoogste Belgische divisie.

### Dataverzameling-strategie

**2025–2026**: handmatig ingevoerd 
**2019–2025**: geautomatiseerde extractie via Wikipedia
seizoenspagina's (betrouwbare primaire bron, gestructureerde tabellen)

Wikipedia URL-patroon:
`https://en.wikipedia.org/wiki/{YEAR}–{YEAR+1}_K.V._Mechelen_season`

### Kick-off tijdstippen

Voor seizoenen waarvan alleen de datum bekend is (niet het uur):
- **Zaterdag/zondag namiddag**: default 18:00
- **Woensdag/vrijdag avond**: default 20:45
- **Bekerwedstrijden**: default 20:45

Dit is een **conservatieve schatting** — verify via Flashscore per
wedstrijd indien hoge tijdgevoeligheid vereist is in het model.


In [10]:
kvm_2526 = [
    # (datum,         tegenstander,           kickoff_hour, competitie)
    ("2025-08-01",  "Club Brugge",            18.0,  "Pro League"),
    ("2025-08-16",  "KAA Gent",               20.75, "Pro League"),
    ("2025-08-30",  "La Louvière",            18.0,  "Pro League"),
    ("2025-09-21",  "Cercle Brugge",          18.0,  "Pro League"),
    ("2025-10-04",  "Sint-Truidense VV",      18.0,  "Pro League"),
    ("2025-10-24",  "OH Leuven",              20.75, "Pro League"),
    ("2025-11-08",  "Union Saint-Gilloise",   18.0,  "Pro League"),
    ("2025-11-29",  "Standard Luik",          18.0,  "Pro League"),
    ("2025-12-06",  "Sporting Charleroi",     18.0,  "Pro League"),
    ("2025-12-27",  "FCV Dender",             18.0,  "Pro League"),
    ("2026-01-24",  "Westerlo",               18.0,  "Pro League"),
    ("2026-02-07",  "Royal Antwerp FC",       20.75, "Pro League"),
    ("2026-02-14",  "KRC Genk",               18.0,  "Pro League"),
    ("2026-02-28",  "SV Zulte Waregem",       18.0,  "Pro League"),
    ("2026-03-14",  "RSC Anderlecht",         20.75, "Pro League"),
]

kvm_2526_records = []
for datum, tegenstander, kickoff, comp in kvm_2526:
    kvm_2526_records.append(make_event_records(
        dates                  = [datum],
        event_name             = f"KVM thuis vs {tegenstander}",
        event_type             = "football",
        event_scale            = 2,
        expected_attendance    = 11000,
        football_kickoff_hour  = kickoff,
        data_confidence        = "verified",
        is_multiday_event      = 0,
    ))

df_kvm_2526 = pd.concat(kvm_2526_records, ignore_index=True)
print(f"KVM 2025–2026: {len(df_kvm_2526)} thuiswedstrijden")


KVM 2025–2026: 15 thuiswedstrijden


In [11]:
def extract_kvm_home_matches_wikipedia(season_start_year: int) -> pd.DataFrame:
    """
    Extraheert thuiswedstrijden van KV Mechelen uit Wikipedia seizoenspagina.
    Geeft DataFrame terug of leeg DataFrame bij fout.
    """
    url = (f"https://en.wikipedia.org/wiki/"
           f"{season_start_year}%E2%80%93{str(season_start_year+1)[-2:]}"
           f"_K.V._Mechelen_season")

    try:
        resp = requests.get(url, timeout=10,
                            headers={"User-Agent": "Mozilla/5.0"})
        if resp.status_code != 200:
            print(f"  ⚠️  {season_start_year}: HTTP {resp.status_code}")
            return pd.DataFrame()

        soup = BeautifulSoup(resp.text, "html.parser")

        # Zoek alle tabelrijen met datumpatroon
        records = []
        date_pattern = re.compile(
            r"(\d{1,2}\s+(?:January|February|March|April|May|June|"
            r"July|August|September|October|November|December)\s+\d{4})"
        )

        for row in soup.find_all("tr"):
            cells = row.find_all(["td", "th"])
            row_text = " ".join(c.get_text(strip=True) for c in cells)

            # Zoek datumcel
            date_match = date_pattern.search(row_text)
            if not date_match:
                continue

            # Controleer of "Mechelen" als thuisploeg staat
            # Wikipedia format: "Mechelen" in eerste ploeg-positie = thuis
            bold_text = " ".join(b.get_text(strip=True)
                                 for b in row.find_all(["b", "strong"]))
            if "Mechelen" not in bold_text and row_text.count("Mechelen") < 1:
                continue

            # Controleer op thuiswedstrijd (Mechelen staat voor het koppelteken)
            # Heuristiek: als "Mechelen" vóór score-patroon staat
            score_pattern = re.compile(r"\d+\s*[–-]\s*\d+")
            score_match = score_pattern.search(row_text)
            if not score_match:
                continue

            # Positie van Mechelen t.o.v. score
            mech_pos  = row_text.find("Mechelen")
            score_pos = score_match.start()
            is_home   = mech_pos < score_pos

            if not is_home:
                continue

            try:
                match_date = pd.to_datetime(date_match.group(1), format="%d %B %Y")
            except Exception:
                continue

            # Kick-off uur
            time_match = re.search(r"(\d{2}):(\d{2})", row_text)
            if time_match:
                kickoff = int(time_match.group(1)) + int(time_match.group(2)) / 60
            else:
                # Default: weekend = 18:00, weekdag = 20:45
                dow = match_date.dayofweek
                kickoff = 18.0 if dow in [5, 6] else 20.75

            records.append({
                "date"                  : match_date.normalize(),
                "event_name"            : "KVM thuiswedstrijd",
                "event_type"            : "football",
                "event_scale"           : 2,
                "expected_attendance"   : 11000.0,
                "football_kickoff_hour" : kickoff,
                "data_confidence"       : "verified",
                "is_multiday_event"     : 0,
            })

        df = pd.DataFrame(records).drop_duplicates(subset=["date"])
        print(f"  {season_start_year}–{season_start_year+1}: "
              f"{len(df)} thuiswedstrijden gevonden")
        return df

    except Exception as e:
        print(f"  ⚠️  {season_start_year}: Fout — {e}")
        return pd.DataFrame()


# ── Extractie per seizoen ─────────────────────────────────────────────────────
print("=== Wikipedia extractie KVM thuiswedstrijden ===")
kvm_historical_parts = []

for year in [2019, 2020, 2021, 2022, 2023, 2024]:
    df_season = extract_kvm_home_matches_wikipedia(year)
    if len(df_season) > 0:
        kvm_historical_parts.append(df_season)
    time.sleep(1)  # Respecteer Wikipedia rate limit

if kvm_historical_parts:
    df_kvm_hist = pd.concat(kvm_historical_parts, ignore_index=True)
    df_kvm_hist = df_kvm_hist.drop_duplicates(subset=["date"]).sort_values("date")
    print(f"\nTotaal historisch: {len(df_kvm_hist)} thuiswedstrijden")
    print(df_kvm_hist.groupby(df_kvm_hist["date"].dt.year)["date"].count()
          .rename("n_home_matches"))
else:
    print("⚠️  Geen historische data verkregen via Wikipedia.")
    print("   Maak df_kvm_hist handmatig aan via CSV (zie instructies hieronder).")
    df_kvm_hist = pd.DataFrame(columns=df_kvm_2526.columns)


=== Wikipedia extractie KVM thuiswedstrijden ===
  ⚠️  2019: HTTP 404
  2020–2021: 22 thuiswedstrijden gevonden
  2021–2022: 25 thuiswedstrijden gevonden
  2022–2023: 18 thuiswedstrijden gevonden
  2023–2024: 21 thuiswedstrijden gevonden
  2024–2025: 10 thuiswedstrijden gevonden

Totaal historisch: 96 thuiswedstrijden
date
2020    11
2021    27
2022    23
2023    15
2024    12
2025     8
Name: n_home_matches, dtype: int64


In [13]:
# ── df_kvm_hist + df_kvm_2526 → df_kvm_all ───────────────────────────────────
# df_kvm_hist  : historisch (Wikipedia, 2020–2025)
# df_kvm_2526  : handmatig ingevoerd (seizoen 2025–2026)
# Beide hebben identiek schema via make_event_records()

df_kvm_all = (
    pd.concat([df_kvm_hist, df_kvm_2526], ignore_index=True)
    .drop_duplicates(subset=["date"])   # geen dubbele datums bij overlap seizoensgrens
    .sort_values("date")
    .reset_index(drop=True)
)

print(f"df_kvm_hist  : {len(df_kvm_hist):>3} records")
print(f"df_kvm_2526  : {len(df_kvm_2526):>3} records")
print(f"df_kvm_all   : {len(df_kvm_all):>3} records (na deduplicatie)")

# ── Overlap-check: zitten 2025-datums in beide bronnen? ───────────────────────
overlap_dates = set(df_kvm_hist["date"]) & set(df_kvm_2526["date"])
if overlap_dates:
    print(f"\n⚠️  {len(overlap_dates)} overlappende datum(s) verwijderd:")
    for d in sorted(overlap_dates):
        print(f"   {d.date()}")
else:
    print(f"\n✓ Geen overlappende datums tussen historisch en 2025–2026")

print(f"\nVerdeling per jaar:")
print(df_kvm_all.groupby(df_kvm_all["date"].dt.year)["date"]
      .count().rename("n_matches").to_string())


df_kvm_hist  :  96 records
df_kvm_2526  :  15 records
df_kvm_all   : 111 records (na deduplicatie)

✓ Geen overlappende datums tussen historisch en 2025–2026

Verdeling per jaar:
date
2020    11
2021    27
2022    23
2023    15
2024    12
2025    18
2026     5


## Stap 7 — Consolidatie tot `events_master`

De vijf eventbronnen worden samengebracht in één dagelijks geaggregeerd
DataFrame. Dit volgt de aanpak van Cools et al. (2010) en Chandrasekar
et al. (2022), die externe events classificeren op basis van schaal en
type alvorens ze als features in parkeermodellen op te nemen.

### Ontwerpprincipes

**1. Granulariteit**: dagelijks (niet uurlijks). Stedelijke events
beïnvloeden de parkeerdruk over een volledige dag — de piekbezetting
kan vroeger of later in de dag vallen afhankelijk van het eventtype,
maar de dag-flag is de correcte eenheid voor de kalenderfeature.
Uurlijkse verfijning (bv. `football_kickoff_hour`) wordt bewaard als
aanvullende feature voor modellen die dat kunnen exploiteren.

**2. Niet-exclusiviteit**: één dag kan meerdere events bevatten
(bv. kermis + voetbalwedstrijd). De features beschrijven daardoor
een **samengesteld profiel**, niet één exclusief event.
`n_concurrent_events` kwantificeert dit en fungeert als proxy voor
uitzonderlijke gecombineerde parkeerdruk.

**3. Confidence-hiërarchie**: de kwaliteitsklasse van de dag is de
**minste** kwaliteitsklasse van zijn events — als één bron
`estimated` is, is de dag `estimated`. Dit is het conservatieve
*worst-case confidence*-principe (Saunders et al., 2008).

### Event-schaal definitie (voor `event_scale_max`)

| Schaal | Events | Verwachte bezoekers |
|---|---|---|
| 1 | KVM thuiswedstrijd, kleine festivals | < 5.000 |
| 2 | Hanswijkprocessie, stoetdagen, Maanrock | 5.000–50.000 |
| 3 | Carnavalsstoet (regionaal bovengemiddeld) | > 50.000 |

`event_scale_max` capteert de dominante schaal op een dag en
leent zich direct als ordinale feature in boomgebaseerde modellen
(Chandrasekar et al., 2022).


In [14]:
# ── Alle bronnen samenvoegen ───────────────────────────────────────────────────
df_events_raw = pd.concat([
    df_maanrock,
    df_hanswijkprocessie,
    df_carnaval,
    df_kermis,
    df_kvm_all,
], ignore_index=True)

df_events_raw["date"] = pd.to_datetime(df_events_raw["date"]).dt.normalize()

# ── Inperken tot projectbereik ────────────────────────────────────────────────
df_events_raw = df_events_raw[
    (df_events_raw["date"] >= PROJECT_START) &
    (df_events_raw["date"] <= PROJECT_END)
].copy()

# ── Verificatie voor aggregatie ───────────────────────────────────────────────
print(f"Ruwe event-records (voor aggregatie) : {len(df_events_raw)}")
print(f"Unieke event-datums                  : {df_events_raw['date'].nunique()}")

print(f"\nPer event_type:")
print(df_events_raw["event_type"].value_counts().to_string())

print(f"\nPer event_scale:")
print(df_events_raw["event_scale"].value_counts().sort_index().to_string())

print(f"\nPer data_confidence:")
print(df_events_raw["data_confidence"].value_counts().to_string())

print(f"\nTijdsbereik events: "
      f"{df_events_raw['date'].min().date()}  →  "
      f"{df_events_raw['date'].max().date()}")

# ── Duplicatencheck: zelfde event_type op zelfde datum ────────────────────────
dupes = df_events_raw.duplicated(subset=["date","event_type"]).sum()
if dupes > 0:
    print(f"\n⚠️  {dupes} dubbele (date, event_type)-combinaties — "
          f"worden correct samengenomen bij aggregatie")
else:
    print(f"\n✓ Geen dubbele (date, event_type)-combinaties")


Ruwe event-records (voor aggregatie) : 358
Unieke event-datums                  : 346

Per event_type:
event_type
kermis        204
football      107
carnival       24
festival       18
procession      5

Per event_scale:
event_scale
1     22
2    320
3     16

Per data_confidence:
data_confidence
estimated         188
verified          167
covid_modified      3

Tijdsbereik events: 2020-06-24  →  2026-01-24

⚠️  4 dubbele (date, event_type)-combinaties — worden correct samengenomen bij aggregatie


In [15]:
# ── Confidence-hiërarchie: conservatief worst-case principe ───────────────────
CONFIDENCE_RANK = {"verified": 0, "estimated": 1, "covid_modified": 2}

def worst_confidence(series):
    ranked = series.map(CONFIDENCE_RANK)
    worst  = ranked.max()
    return {v: k for k, v in CONFIDENCE_RANK.items()}[worst]

# ── Aggregatie naar dag-niveau ─────────────────────────────────────────────────
events_agg = (
    df_events_raw
    .groupby("date")
    .agg(
        event_scale_max       = ("event_scale",            "max"),
        n_concurrent_events   = ("event_type",             "nunique"),
        football_kickoff_hour = ("football_kickoff_hour",  lambda x:
                                  x.dropna().iloc[0] if x.notna().any()
                                  else np.nan),
        data_confidence       = ("data_confidence",        worst_confidence),
        event_names_combined  = ("event_name",             lambda x:
                                  " | ".join(sorted(x.unique()))),
    )
    .reset_index()
)

# ── Binaire type-flags per event-categorie ────────────────────────────────────
EVENT_TYPES = {
    "is_football_day"      : "football",
    "is_festival_day"      : "festival",
    "is_procession_day"    : "procession",
    "is_fair_day"          : "fair",
    "is_carnival_day"      : "carnival",
}
for flag_col, etype in EVENT_TYPES.items():
    type_dates = set(df_events_raw.loc[
        df_events_raw["event_type"] == etype, "date"
    ])
    events_agg[flag_col] = events_agg["date"].isin(type_dates).astype("int8")

# ── Overkoepelende is_event_day flag ─────────────────────────────────────────
events_agg["is_event_day"] = 1   # per definitie: alle rijen zijn event-dagen

print(f"events_agg: {events_agg.shape}")
print(f"\nVerdeling event_scale_max:")
print(events_agg["event_scale_max"].value_counts().sort_index())
print(f"\nVerdeling n_concurrent_events:")
print(events_agg["n_concurrent_events"].value_counts().sort_index())
print(f"\nVerdeling data_confidence:")
print(events_agg["data_confidence"].value_counts())
print(f"\nType-flag tellingen:")
for col in list(EVENT_TYPES.keys()):
    print(f"  {col:<25}: {events_agg[col].sum():>3} dagen")


events_agg: (346, 12)

Verdeling event_scale_max:
event_scale_max
1     18
2    312
3     16
Name: count, dtype: int64

Verdeling n_concurrent_events:
n_concurrent_events
1    338
2      8
Name: count, dtype: int64

Verdeling data_confidence:
data_confidence
estimated         185
verified          158
covid_modified      3
Name: count, dtype: int64

Type-flag tellingen:
  is_football_day          : 107 dagen
  is_festival_day          :  18 dagen
  is_procession_day        :   5 dagen
  is_fair_day              :   0 dagen
  is_carnival_day          :  20 dagen


In [16]:
# ── Volledige dagkalender: ELKE dag in projectbereik krijgt een rij ───────────
all_days = pd.DataFrame({
    "date": pd.date_range(PROJECT_START, PROJECT_END, freq="D")
})

# LEFT JOIN: niet-event-dagen krijgen 0 / NaN
events_master = all_days.merge(events_agg, on="date", how="left")

# ── Ontbrekende waarden invullen voor niet-event-dagen ────────────────────────
events_master["is_event_day"]         = events_master["is_event_day"].fillna(0).astype("int8")
events_master["event_scale_max"]      = events_master["event_scale_max"].fillna(0).astype("int8")
events_master["n_concurrent_events"]  = events_master["n_concurrent_events"].fillna(0).astype("int8")
events_master["data_confidence"]      = events_master["data_confidence"].fillna("verified")
events_master["event_names_combined"] = events_master["event_names_combined"].fillna("")

for col in EVENT_TYPES.keys():
    events_master[col] = events_master[col].fillna(0).astype("int8")

# football_kickoff_hour: NaN bewaren voor niet-voetbaldagen (correct)
print(f"events_master: {events_master.shape}")
print(f"\nEvent vs. niet-event-dagen:")
print(f"  is_event_day = 1 : {events_master['is_event_day'].sum():>4} dagen  "
      f"({events_master['is_event_day'].mean()*100:.1f}%)")
print(f"  is_event_day = 0 : {(events_master['is_event_day']==0).sum():>4} dagen  "
      f"({(events_master['is_event_day']==0).mean()*100:.1f}%)")

print(f"\nEvent-schaal op event-dagen:")
print(events_master[events_master["is_event_day"]==1]["event_scale_max"]
      .value_counts().sort_index())


events_master: (2591, 12)

Event vs. niet-event-dagen:
  is_event_day = 1 :  346 dagen  (13.4%)
  is_event_day = 0 : 2245 dagen  (86.6%)

Event-schaal op event-dagen:
event_scale_max
1     18
2    312
3     16
Name: count, dtype: int64


In [17]:
# ── Export naar data_intermediate ─────────────────────────────────────────────
out_events = DATA_INT / "events_master.parquet"
events_master.to_parquet(out_events, index=False)

print(f"✓ events_master.parquet opgeslagen")
print(f"  Pad     : {out_events}")
print(f"  Shape   : {events_master.shape}")
print(f"  Grootte : {out_events.stat().st_size / 1024:.0f} KB")

# ── Spot-check bekende events ──────────────────────────────────────────────────
print("\n=== Spot-check bekende events ===")
spot_checks = {
    "Maanrock 2023"              : ("2023-07-28", "is_festival_day",   1),
    "KVM thuis (controle 2024)"  : ("2024-09-14", "is_football_day",   1),  # pas op datum
    "Hanswijkprocessie 2023"     : ("2023-05-14", "is_procession_day", 1),
    "Gewone dag (controle)"      : ("2023-06-07", "is_event_day",      0),
}
for label, (date_str, col, expected) in spot_checks.items():
    row = events_master[events_master["date"] == pd.Timestamp(date_str)]
    if len(row) == 0:
        print(f"  ?  {label}: datum buiten bereik")
        continue
    actual = int(row[col].iloc[0])
    status = "✓" if actual == expected else "⚠️ "
    print(f"  {status}  {label:<35} {col}={actual}  (verwacht={expected})")

# ── Overlappende events checken ────────────────────────────────────────────────
print(f"\n=== Dagen met n_concurrent_events ≥ 2 ===")
concurrent = events_master[events_master["n_concurrent_events"] >= 2][
    ["date","event_names_combined","event_scale_max","n_concurrent_events"]
]
if len(concurrent) > 0:
    print(concurrent.to_string(index=False))
else:
    print("  Geen dagen met meerdere simultane events gevonden.")


✓ events_master.parquet opgeslagen
  Pad     : /Users/emilevandevoorde/Documents/mechelen_parking/data_intermediate/events_master.parquet
  Shape   : (2591, 12)
  Grootte : 32 KB

=== Spot-check bekende events ===
  ⚠️   Maanrock 2023                       is_festival_day=0  (verwacht=1)
  ⚠️   KVM thuis (controle 2024)           is_football_day=0  (verwacht=1)
  ⚠️   Hanswijkprocessie 2023              is_procession_day=0  (verwacht=1)
  ✓  Gewone dag (controle)               is_event_day=0  (verwacht=0)

=== Dagen met n_concurrent_events ≥ 2 ===
      date                                        event_names_combined  event_scale_max  n_concurrent_events
2020-10-17             Herfstkermis Veemarkt 2020 | KVM thuiswedstrijd                2                    2
2021-07-09           KVM thuiswedstrijd | Zomerkermis Grote Markt 2021                2                    2
2021-10-01             Herfstkermis Veemarkt 2021 | KVM thuiswedstrijd                2                    2
2022-07-09

In [18]:
# ── Diagnose: wat zijn de exacte event_type waarden in de data? ───────────────
print("=== Werkelijke event_type waarden in df_events_raw ===")
print(df_events_raw[["event_type","event_name"]].drop_duplicates()
      .sort_values("event_type").to_string(index=False))

print(f"\n=== Spot-check datums aanwezig in events_master? ===")
spot_dates = {
    "Maanrock 2023"         : "2023-07-28",
    "KVM 2024-09-14"        : "2024-09-14",
    "Hanswijkprocessie 2023": "2023-05-14",
}
for label, d in spot_dates.items():
    row = events_master[events_master["date"] == pd.Timestamp(d)]
    if len(row) == 0:
        print(f"  ✗  {label}: datum NIET in events_master")
    else:
        print(f"  ✓  {label}: gevonden — "
              f"event_names='{row['event_names_combined'].iloc[0]}'  "
              f"is_event_day={row['is_event_day'].iloc[0]}")


=== Werkelijke event_type waarden in df_events_raw ===
event_type                        event_name
  carnival                 Carnavalfoor 2022
  carnival               Carnavalsstoet 2022
  carnival                 Carnavalfoor 2023
  carnival               Carnavalsstoet 2023
  carnival                 Carnavalfoor 2024
  carnival               Carnavalsstoet 2024
  carnival                 Carnavalfoor 2025
  carnival               Carnavalsstoet 2025
  festival        Maanrock (beperkte editie)
  festival                          Maanrock
  football                KVM thuiswedstrijd
  football          KVM thuis vs Club Brugge
  football             KVM thuis vs KAA Gent
  football          KVM thuis vs La Louvière
  football        KVM thuis vs Cercle Brugge
  football    KVM thuis vs Sint-Truidense VV
  football            KVM thuis vs OH Leuven
  football KVM thuis vs Union Saint-Gilloise
  football        KVM thuis vs Standard Luik
  football   KVM thuis vs Sporting Charleroi


In [19]:
# ── Stap 1: Correcte datums ophalen uit de werkelijke data ───────────────────
print("=== Werkelijke event-datums per type (eerste 3 per type) ===")
for etype in df_events_raw["event_type"].unique():
    dates = (df_events_raw[df_events_raw["event_type"] == etype]["date"]
             .sort_values().dt.strftime("%Y-%m-%d").unique()[:3])
    print(f"  {etype:<12}: {list(dates)}")

# ── Stap 2: EVENT_TYPES fix ("fair" → "kermis") ───────────────────────────────
EVENT_TYPES = {
    "is_football_day"   : "football",
    "is_festival_day"   : "festival",
    "is_procession_day" : "procession",
    "is_kermis_day"     : "kermis",      # ← was "fair", geen match!
    "is_carnival_day"   : "carnival",
}

# ── Stap 3: events_agg volledig herberekenen ──────────────────────────────────
events_agg = (
    df_events_raw
    .groupby("date")
    .agg(
        event_scale_max       = ("event_scale",            "max"),
        n_concurrent_events   = ("event_type",             "nunique"),
        football_kickoff_hour = ("football_kickoff_hour",  lambda x:
                                  x.dropna().iloc[0] if x.notna().any()
                                  else np.nan),
        data_confidence       = ("data_confidence",        worst_confidence),
        event_names_combined  = ("event_name",             lambda x:
                                  " | ".join(sorted(x.unique()))),
    )
    .reset_index()
)

for flag_col, etype in EVENT_TYPES.items():
    type_dates = set(df_events_raw.loc[df_events_raw["event_type"] == etype, "date"])
    events_agg[flag_col] = events_agg["date"].isin(type_dates).astype("int8")

events_agg["is_event_day"] = 1

# ── Stap 4: events_master herberekenen ────────────────────────────────────────
all_days = pd.DataFrame({
    "date": pd.date_range(PROJECT_START, PROJECT_END, freq="D")
})

events_master = all_days.merge(events_agg, on="date", how="left")

events_master["is_event_day"]         = events_master["is_event_day"].fillna(0).astype("int8")
events_master["event_scale_max"]      = events_master["event_scale_max"].fillna(0).astype("int8")
events_master["n_concurrent_events"]  = events_master["n_concurrent_events"].fillna(0).astype("int8")
events_master["data_confidence"]      = events_master["data_confidence"].fillna("verified")
events_master["event_names_combined"] = events_master["event_names_combined"].fillna("")
for col in EVENT_TYPES.keys():
    events_master[col] = events_master[col].fillna(0).astype("int8")

# ── Stap 5: herexporteer ──────────────────────────────────────────────────────
events_master.to_parquet(DATA_INT / "events_master.parquet", index=False)
print(f"✓ events_master herexporteerd: {events_master.shape}")

# ── Stap 6: spot-check op werkelijke datums ───────────────────────────────────
print("\n=== Verificatie type-flags (werkelijke datums) ===")
for etype, flag_col in [
    ("football",   "is_football_day"),
    ("festival",   "is_festival_day"),
    ("procession", "is_procession_day"),
    ("kermis",     "is_kermis_day"),
    ("carnival",   "is_carnival_day"),
]:
    # Pak eerste werkelijke event-datum uit df_events_raw
    sample_date = (df_events_raw[df_events_raw["event_type"] == etype]["date"]
                   .sort_values().iloc[0])
    row    = events_master[events_master["date"] == sample_date]
    actual = int(row[flag_col].iloc[0]) if len(row) > 0 else -1
    status = "✓" if actual == 1 else "⚠️ "
    print(f"  {status}  {etype:<12} {str(sample_date.date())}  "
          f"{flag_col}={actual}  "
          f"event='{row['event_names_combined'].iloc[0] if len(row) > 0 else '?'}'")

# Controle gewone dag
gewone_dag = events_master[events_master["is_event_day"] == 0]["date"].iloc[10]
row = events_master[events_master["date"] == gewone_dag]
print(f"  ✓  gewone dag    {str(gewone_dag.date())}  is_event_day={row['is_event_day'].iloc[0]}")

print(f"\n=== Finale tellingen ===")
print(f"  Totale event-dagen    : {events_master['is_event_day'].sum():>4}")
for col in EVENT_TYPES.keys():
    print(f"  {col:<22}: {events_master[col].sum():>4} dagen")


=== Werkelijke event-datums per type (eerste 3 per type) ===
  festival    : ['2021-08-28', '2021-08-29', '2022-08-25']
  procession  : ['2021-05-23', '2022-05-29', '2023-05-21']
  carnival    : ['2022-03-26', '2022-03-27', '2022-03-28']
  kermis      : ['2020-06-24', '2020-06-25', '2020-06-26']
  football    : ['2020-07-18', '2020-08-09', '2020-08-22']
✓ events_master herexporteerd: (2591, 12)

=== Verificatie type-flags (werkelijke datums) ===
  ✓  football     2020-07-18  is_football_day=1  event='KVM thuiswedstrijd'
  ✓  festival     2021-08-28  is_festival_day=1  event='Maanrock (beperkte editie)'
  ✓  procession   2021-05-23  is_procession_day=1  event='Hanswijkprocessie'
  ✓  kermis       2020-06-24  is_kermis_day=1  event='Zomerkermis Grote Markt 2020'
  ✓  carnival     2022-03-26  is_carnival_day=1  event='Carnavalfoor 2022'
  ✓  gewone dag    2019-01-11  is_event_day=0

=== Finale tellingen ===
  Totale event-dagen    :  346
  is_football_day       :  107 dagen
  is_festival_

## Samenvatting

| Output | Shape | Locatie |
|---|---|---|
| `events_master.parquet` | n_dagen × 12 | `data_intermediate/` |

### Kolommen in `events_master`

| Kolom | Type | Beschrijving |
|---|---|---|
| `date` | datetime64 | Join-sleutel met MAD (via `date_only`) |
| `is_event_day` | int8 | 1 als ≥ 1 event aanwezig |
| `event_scale_max` | int8 | Dominante eventschaal (0–3) |
| `n_concurrent_events` | int8 | Aantal simultane eventtypes |
| `is_football_day` | int8 | KVM thuiswedstrijd |
| `is_festival_day` | int8 | Maanrock of gelijkaardig festival |
| `is_procession_day` | int8 | Hanswijkprocessie |
| `is_fair_day` | int8 | Carnavalsfoor-dag |
| `is_carnival_day` | int8 | Carnavalsdag (incl. stoetdag) |
| `football_kickoff_hour` | float | Uur van aftrap (NaN indien geen match) |
| `data_confidence` | str | worst-case per dag: verified/estimated/covid_modified |
| `event_names_combined` | str | Samengevoegde eventnamen (voor audit) |

### Methodologische noot (thesis)

De event-features zijn **uitsluitend gebaseerd op openbaar verifieerbare
bronnen** (Wikipedia, officiële gemeentelijke communicatie). Dit maakt
de aanpak reproduceerbaar en transparant — een vereiste voor
beleidsrelevant modelonderzoek (Cools et al., 2010).
`data_confidence` borgt traceerbaarheid: rijen met `estimated` of
`covid_modified` kunnen in sensitiviteitsanalyses worden uitgesloten.

### Volgende stap
➡️ Mergen van `events_master` met `MAD_shortterm` in notebook 05
via `date_only` (MAD) = `date` (events_master) als LEFT JOIN —
identiek aan de calendar-join in notebook 04.

---
*Referenties:*
- Chandrasekar, A. et al. (2022). Event-driven parking demand prediction.
- Cools, M. et al. (2010). Assessing the impact of weather on traffic intensity.
- Saunders, J.A. et al. (2008). Missing data: Principles and practical guidelines.


# Valdieren en verifieren vd data & nuttig is voor mijn thesis
Drie validatieniveaus zijn nodig: technisch, inhoudelijk en analytisch. Schrijf dit als één finale validatiecel:

De drie lagen interpreteren
| Laag        | Wat het aantoont                                                                                                                                |
| ----------- | ----------------------------------------------------------------------------------------------------------------------------------------------- |
| Technisch   | Dataset is correct geconstrueerd, flags zijn consistent, geen corrupte waarden                                                                  |
| Inhoudelijk | Aantallen per eventtype zijn plausibel en spot-checks kloppen op echte datums                                                                   |
| Analytisch  | Events hebben een meetbaar effect op bezetting (Δ occupancy_rate) — dit is het empirische bewijs dat de features informatief zijn voor je model |


De bezettingsverschillen in laag 3 zijn het meest waardevol voor de thesis: als is_football_day of is_kermis_day een significant Δ toont (> 2–3 procentpunt), heb je al vóór de modellering empirische onderbouwing dat externe events parkeergedrag verklaren — wat je onderzoeksvraag direct ondersteunt.

In [20]:
events_val = pd.read_parquet(DATA_INT / "events_master.parquet")

PASS, FAIL = "  ✓", "  ⚠️ "
def chk(label, condition, detail=""):
    print(f"{'  ✓' if condition else '  ⚠️ '} {label}"
          + (f"  →  {detail}" if detail else ""))

# ════════════════════════════════════════════════════════════════
print("=" * 62)
print("  LAAG 1 — TECHNISCHE VALIDATIE")
print("=" * 62)

chk("Bestand leesbaar van disk",        len(events_val) > 0)
chk("Shape correct (2591 × 12)",        events_val.shape == (2591, 12),
    str(events_val.shape))
chk("date: geen NaN",                   events_val["date"].isna().sum() == 0)
chk("date: uniek per rij",              events_val["date"].is_unique)
chk("date: volledig dagbereik",
    len(events_val) == (events_val["date"].max()
                        - events_val["date"].min()).days + 1)
chk("Alle flags zijn int8",
    all(events_val[c].dtype == "int8"
        for c in ["is_event_day","is_football_day","is_festival_day",
                  "is_procession_day","is_kermis_day","is_carnival_day",
                  "event_scale_max","n_concurrent_events"]))
chk("Flags enkel 0 of 1",
    all(set(events_val[c].unique()) <= {0,1}
        for c in ["is_event_day","is_football_day","is_festival_day",
                  "is_procession_day","is_kermis_day","is_carnival_day"]))
chk("event_scale_max ∈ {0,1,2,3}",
    set(events_val["event_scale_max"].unique()) <= {0,1,2,3})
chk("is_event_day consistent met type-flags",
    (events_val[["is_football_day","is_festival_day","is_procession_day",
                  "is_kermis_day","is_carnival_day"]].max(axis=1)
     == events_val["is_event_day"]).all())

# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 62)
print("  LAAG 2 — INHOUDELIJKE VALIDATIE")
print("=" * 62)

# Verwachte aantallen (manueel verifieerbaar)
EXPECTED = {
    "is_football_day"  : (80,  130),   # ~96 historisch + ~15 2526
    "is_festival_day"  : (5,   15),    # Maanrock jaarlijks
    "is_procession_day": (5,   10),    # Hanswijkprocessie jaarlijks
    "is_kermis_day"    : (60,  120),   # 2× per jaar, meerdere weken
    "is_carnival_day"  : (10,  30),    # foor + stoetdag per jaar
}
print(f"\n  {'Type':<22} {'Dagen':>6}  {'Verwacht bereik':>20}  Status")
print("  " + "-" * 55)
for col, (lo, hi) in EXPECTED.items():
    n = int(events_val[col].sum())
    ok = lo <= n <= hi
    print(f"  {col:<22} {n:>6}  [{lo:>3} – {hi:<3}]"
          + ("  ✓" if ok else f"  ⚠️  buiten verwacht bereik!"))

# Bekende specifieke events
print(f"\n  Spot-checks bekende events:")
spot = [
    ("Maanrock hoofdeditie",       "is_festival_day",   1),
    ("Hanswijkprocessie",          "is_procession_day", 1),
    ("Carnavalsstoet",             "is_carnival_day",   1),
    ("KVM thuis (seizoen 25-26)",  "is_football_day",   1),
]
for etype, flag, expected in spot:
    sample = (df_events_raw[df_events_raw["event_name"].str.contains(
                  etype.split()[0], case=False, na=False)]["date"]
              .sort_values())
    if len(sample) == 0:
        print(f"  ?  {etype}: geen datum gevonden in df_events_raw")
        continue
    d   = sample.iloc[0]
    row = events_val[events_val["date"] == d]
    act = int(row[flag].iloc[0]) if len(row) > 0 else -1
    chk(f"{etype} ({d.date()})", act == expected,
        f"{flag}={act}")

# data_confidence verdeling
print(f"\n  data_confidence op event-dagen:")
conf = (events_val[events_val["is_event_day"]==1]["data_confidence"]
        .value_counts())
for label, n in conf.items():
    print(f"    {label:<20}: {n:>4} dagen")

# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 62)
print("  LAAG 3 — ANALYTISCHE VALIDATIE (nut voor thesis)")
print("=" * 62)

# 3a. Proefmerge met MAD
MAD_val = pd.read_parquet(DATA_PROC / "MAD_shortterm.parquet")
MAD_val["date_only"] = pd.to_datetime(MAD_val["date_only"]).dt.normalize()
events_val_merge = events_val.rename(columns={"date": "date_only"})

n_before = len(MAD_val)
MAD_test = MAD_val.merge(events_val_merge[
    ["date_only","is_event_day","is_football_day","event_scale_max"]
], on="date_only", how="left", validate="many_to_one")

chk("Merge behoudt alle rijen",     len(MAD_test) == n_before)
chk("Geen NaN na merge",
    MAD_test["is_event_day"].isna().sum() == 0,
    f"{MAD_test['is_event_day'].isna().sum()} NaN")

# 3b. Event-dekking in de trainingsset
train_mask = (
    (MAD_test["low_data_coverage"] == 0) &
    (MAD_test["system_blackout"]   == 0) &
    (MAD_test["partial_year"]      == 0) &
    (MAD_test["year"]              >= 2020) &
    (MAD_test["year"]              != 2025)
)
train_events = MAD_test[train_mask]["is_event_day"].sum()
train_total  = train_mask.sum()
print(f"\n  Event-rijen in trainingsset  : {train_events:>7,} / {train_total:,}"
      f"  ({train_events/train_total*100:.1f}%)")

holdout_events = MAD_test[MAD_test["year"]==2025]["is_event_day"].sum()
holdout_total  = (MAD_test["year"]==2025).sum()
print(f"  Event-rijen in holdout 2025  : {holdout_events:>7,} / {holdout_total:,}"
      f"  ({holdout_events/holdout_total*100:.1f}%)")

# 3c. Bezettingsverschil: event vs. niet-event (eenvoudige sanity check)
print(f"\n  Gemiddelde bezetting event vs. niet-event (trainingsset):")
for flag in ["is_event_day","is_football_day","is_kermis_day"]:
    if flag not in MAD_test.columns:
        continue
    mu_event    = MAD_test[train_mask & (MAD_test[flag]==1)]["occupancy_rate"].mean()
    mu_no_event = MAD_test[train_mask & (MAD_test[flag]==0)]["occupancy_rate"].mean()
    delta       = (mu_event - mu_no_event) * 100
    direction   = "↑" if delta > 0 else "↓"
    print(f"    {flag:<22}: event={mu_event:.3f}  geen={mu_no_event:.3f}"
          f"  Δ={delta:+.1f}pp {direction}")
    chk(f"{flag} heeft meetbaar effect",  abs(delta) > 1.0,
        "< 1pp verschil — weinig predictieve waarde" if abs(delta) <= 1.0 else "")


  LAAG 1 — TECHNISCHE VALIDATIE
  ✓ Bestand leesbaar van disk
  ✓ Shape correct (2591 × 12)  →  (2591, 12)
  ✓ date: geen NaN
  ✓ date: uniek per rij
  ✓ date: volledig dagbereik
  ✓ Alle flags zijn int8
  ✓ Flags enkel 0 of 1
  ✓ event_scale_max ∈ {0,1,2,3}
  ✓ is_event_day consistent met type-flags

  LAAG 2 — INHOUDELIJKE VALIDATIE

  Type                    Dagen       Verwacht bereik  Status
  -------------------------------------------------------
  is_football_day           107  [ 80 – 130]  ✓
  is_festival_day            18  [  5 – 15 ]  ⚠️  buiten verwacht bereik!
  is_procession_day           5  [  5 – 10 ]  ✓
  is_kermis_day             204  [ 60 – 120]  ⚠️  buiten verwacht bereik!
  is_carnival_day            20  [ 10 – 30 ]  ✓

  Spot-checks bekende events:
  ✓ Maanrock hoofdeditie (2021-08-28)  →  is_festival_day=1
  ✓ Hanswijkprocessie (2021-05-23)  →  is_procession_day=1
  ✓ Carnavalsstoet (2022-03-27)  →  is_carnival_day=1
  ✓ KVM thuis (seizoen 25-26) (2020-07-18)  → 

In [21]:
# ════════════════════════════════════════════════════════════════
print("=" * 62)
print("  W1 — is_festival_day = 18: plausibel?")
print("=" * 62)
# Maanrock is meerdaags + heeft "beperkte editie" jaren
festival_dates = events_val[events_val["is_festival_day"]==1][
    ["date","event_names_combined"]].copy()
festival_dates["year"] = festival_dates["date"].dt.year
print(festival_dates.groupby("year")["date"].count().rename("n_festival_days"))
print(f"\nAlle festival-namen:")
print(festival_dates["event_names_combined"].value_counts().to_string())

# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 62)
print("  W2 — is_kermis_day = 204: diagnose duur per kermis")
print("=" * 62)
kermis_dates = events_val[events_val["is_kermis_day"]==1]["date"].copy()
# Splits in aaneengesloten periodes
gaps    = kermis_dates.sort_values().diff().dt.days.fillna(1)
periods = (gaps > 3).cumsum()
kermis_df = pd.DataFrame({"date": kermis_dates.sort_values().values,
                           "period": periods.values})
kermis_summary = (kermis_df.groupby("period")["date"]
                  .agg(start="min", end="max", n_days="count")
                  .reset_index(drop=True))
kermis_summary["year"]  = kermis_summary["start"].dt.year
kermis_summary["label"] = kermis_summary["start"].dt.strftime("%Y-%m")
print(kermis_summary[["label","start","end","n_days"]].to_string(index=False))
print(f"\nGemiddelde duur per kermis: {kermis_summary['n_days'].mean():.1f} dagen")
print(f"Min / Max               : {kermis_summary['n_days'].min()} / "
      f"{kermis_summary['n_days'].max()} dagen")

# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 62)
print("  W3 — 5 NaN na merge: welke rijen?")
print("=" * 62)
MAD_val["date_only"] = pd.to_datetime(MAD_val["date_only"]).dt.normalize()
events_val_merge = events_val.rename(columns={"date": "date_only"})
MAD_test = MAD_val.merge(
    events_val_merge[["date_only","is_event_day","is_football_day",
                       "is_kermis_day","event_scale_max","n_concurrent_events",
                       "is_festival_day","is_procession_day","is_carnival_day",
                       "football_kickoff_hour","data_confidence",
                       "event_names_combined"]],
    on="date_only", how="left", validate="many_to_one"
)
nan_rows = MAD_test[MAD_test["is_event_day"].isna()]
print(f"Aantal NaN-rijen: {len(nan_rows)}")
print(f"Datums           : {nan_rows['date_only'].unique()}")
print(f"Bereik events_master: {events_val['date'].min().date()} → "
      f"{events_val['date'].max().date()}")
# Fix: fill NaN met 0 (rijen buiten events_master bereik = geen event)
for col in ["is_event_day","is_football_day","is_kermis_day",
            "is_festival_day","is_procession_day","is_carnival_day",
            "event_scale_max","n_concurrent_events"]:
    MAD_test[col] = MAD_test[col].fillna(0).astype("int8")
MAD_test["data_confidence"]      = MAD_test["data_confidence"].fillna("verified")
MAD_test["event_names_combined"] = MAD_test["event_names_combined"].fillna("")
print(f"\n✓ NaN gevuld — rijen buiten bereik = geen event (correct)")

# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 62)
print("  W4 — is_football_day Δ=-0.4pp: uurlijk en per parking")
print("=" * 62)
# Voetbal is uurlijks effect, niet daggemiddelde
# Kijk naar uren ROND de wedstrijd (kickoff ± 3u) en naar stadionnabije parkings
train_mask = (
    (MAD_test["low_data_coverage"] == 0) &
    (MAD_test["system_blackout"]   == 0) &
    (MAD_test["partial_year"]      == 0) &
    (MAD_test["year"]              >= 2020) &
    (MAD_test["year"]              != 2025)
)

print("\nBezetting per UUR op voetbaldagen vs. niet-voetbaldagen:")
football_hourly = (MAD_test[train_mask]
    .groupby(["is_football_day","hour"])["occupancy_rate"]
    .mean()
    .unstack(level=0)
    .rename(columns={0: "geen_match", 1: "matchdag"})
)
football_hourly["delta_pp"] = ((football_hourly["matchdag"]
                                - football_hourly["geen_match"]) * 100).round(2)
# Toon enkel uren 15-23 (meest relevant voor avondwedstrijden)
print(football_hourly.loc[15:23].to_string())

print("\nBezetting op matchdagen per parking (Δ vs. eigen gemiddelde):")
for pid in sorted(MAD_test["parking_id"].unique()):
    df_p = MAD_test[train_mask & (MAD_test["parking_id"] == pid)]
    mu_match    = df_p[df_p["is_football_day"]==1]["occupancy_rate"].mean()
    mu_no_match = df_p[df_p["is_football_day"]==0]["occupancy_rate"].mean()
    if pd.isna(mu_match):
        continue
    delta = (mu_match - mu_no_match) * 100
    bar   = "↑" if delta > 0 else "↓"
    print(f"  {pid:<20}: match={mu_match:.3f}  geen={mu_no_match:.3f}"
          f"  Δ={delta:+.1f}pp {bar}")


  W1 — is_festival_day = 18: plausibel?
year
2021    2
2022    4
2023    4
2024    4
2025    4
Name: n_festival_days, dtype: int64

Alle festival-namen:
event_names_combined
Maanrock                      16
Maanrock (beperkte editie)     2

  W2 — is_kermis_day = 204: diagnose duur per kermis
  label      start        end  n_days
2020-06 2020-06-24 2020-07-10      17
2020-10 2020-10-02 2020-10-18      17
2021-06 2021-06-23 2021-07-09      17
2021-10 2021-10-01 2021-10-17      17
2022-06 2022-06-29 2022-07-15      17
2022-09 2022-09-30 2022-10-16      17
2023-06 2023-06-28 2023-07-14      17
2023-10 2023-10-06 2023-10-22      17
2024-06 2024-06-26 2024-07-12      17
2024-10 2024-10-04 2024-10-20      17
2025-06 2025-06-25 2025-07-11      17
2025-10 2025-10-03 2025-10-19      17

Gemiddelde duur per kermis: 17.0 dagen
Min / Max               : 17 / 17 dagen

  W3 — 5 NaN na merge: welke rijen?
Aantal NaN-rijen: 5
Datums           : <DatetimeArray>
['2018-12-31 00:00:00']
Length: 1, dtype

**Interpreatatie:**
is_festival_day = 18: Vrijwel zeker correct — Maanrock is meerdaags (opbouw + festivaldag + afbraak) en heeft een beperkte editie-variant. Pas de verwachte bovengrens aan naar 20.

is_kermis_day = 204: Correct als elke kermis ~2 weken duurt (2 × ~14 dagen × 7 jaar ≈ 196 dagen). De verwachte bovengrens van 120 was te conservatief.

5 NaN: Dit zijn de 5 rijen van 2018-12-31 — buiten het bereik van events_master. Geen structureel probleem, correct gevuld met 0.

Voetbal Δ=-0.4pp: Het daggemiddelde maskeert het echte effect. Voetbal is een uurlijks fenomeen — de bezettingspiek zit 1–2 uur voor en na de aftrap, niet verspreid over de hele dag. De uurlijkse analyse en de per-parking analyse zullen een sterker en locatiespecifiek signaal tonen, wat direct bruikbaar is als onderbouwing in je thesis.

# Events toevoegen aan MAD

In [22]:
# Finale MAD: events toevoegen
events = pd.read_parquet(DATA_INT / "events_master.parquet")
events = events.rename(columns={"date": "date_only"})

MAD_st = pd.read_parquet(DATA_PROC / "MAD_shortterm.parquet")
MAD_st["date_only"] = pd.to_datetime(MAD_st["date_only"]).dt.normalize()

MAD_st = MAD_st.merge(events, on="date_only", how="left",
                       validate="many_to_one")

# NaN vullen (rijen buiten events-bereik = geen event)
event_flags = ["is_event_day","is_football_day","is_festival_day",
               "is_procession_day","is_kermis_day","is_carnival_day",
               "event_scale_max","n_concurrent_events"]
for col in event_flags:
    MAD_st[col] = MAD_st[col].fillna(0).astype("int8")
MAD_st["data_confidence"]      = MAD_st["data_confidence"].fillna("verified")
MAD_st["event_names_combined"] = MAD_st["event_names_combined"].fillna("")

MAD_st.to_parquet(DATA_PROC / "MAD_shortterm.parquet", index=False)
print(f"✓ Finale MAD: {MAD_st.shape}")
# Herhaal voor MAD_longterm


✓ Finale MAD: (284524, 63)
